In [3]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import time, os, json
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from thop import profile as thop_profile

# ============================================================
# Evaluation-only script — loads the ALREADY TRAINED BranchyNet
# weights produced by the CIFAR-10 BranchyNet training script
# (branchynet_resnet50_cifar10_corrected.pth) and recomputes:
#   - final-exit accuracy / precision / recall / f1
#   - the early-exit threshold sweep (adaptive_forward)
#   - FLOPs, params, disk size, CPU/GPU inference timing
# No training happens here.
#
# IMPORTANT: this uses the BranchyResNet50 architecture (modified
# 3x3/stride-1 stem, no maxpool, + branch1/branch2 + final fc).
# It is NOT compatible with plain ResNet-50 baseline weights
# (__2__baseline_resnet50_cifar10.pth) — use the original baseline
# eval script for that one instead.
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CIFAR_MEAN  = (0.4914, 0.4822, 0.4465)
CIFAR_STD   = (0.2023, 0.1994, 0.2010)
NUM_CLASSES = 10

# Path to the weights saved by the BranchyNet training script
SAVE_PATH = "branchynet_resnet50_cifar10_corrected.pth"

# Exit confidence thresholds to sweep during adaptive evaluation
EXIT_THRESHOLDS = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]

print(f"Using device: {DEVICE}")

# ── AUXILIARY BRANCH (must match training architecture exactly) ──
class EarlyExitBranch(nn.Module):
    def __init__(self, input_channels: int, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.branch = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.BatchNorm1d(input_channels),
            nn.Linear(input_channels, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.branch(x)

# ── BRANCHYNET MODEL (must match training architecture exactly) ──
class BranchyResNet50(nn.Module):
    """
    ResNet-50 adapted for CIFAR-10 (3x3/stride-1 stem, no maxpool)
    with two auxiliary early-exit branches after layer1 and layer2.
    """
    def __init__(self, num_classes: int = NUM_CLASSES, pretrained: bool = False):
        super().__init__()

        backbone = models.resnet50(weights=None) if not pretrained else models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V1)
        backbone.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1,
                                     padding=1, bias=False)
        backbone.maxpool = nn.Identity()
        backbone.fc      = nn.Linear(backbone.fc.in_features, num_classes)

        self.stem    = nn.Sequential(backbone.conv1, backbone.bn1,
                                     backbone.relu, backbone.maxpool)
        self.layer1  = backbone.layer1   # out: 256 ch
        self.layer2  = backbone.layer2   # out: 512 ch
        self.layer3  = backbone.layer3   # out: 1024 ch
        self.layer4  = backbone.layer4   # out: 2048 ch
        self.avgpool = backbone.avgpool
        self.fc      = backbone.fc

        self.branch1 = EarlyExitBranch(256, num_classes)
        self.branch2 = EarlyExitBranch(512, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        logits1 = self.branch1(x)
        x = self.layer2(x)
        logits2 = self.branch2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        logits3 = self.fc(x)
        return logits1, logits2, logits3

    @torch.no_grad()
    def adaptive_forward(self, x, threshold: float = 0.8):
        B = x.size(0)
        x = self.stem(x)

        x = self.layer1(x)
        logits1 = self.branch1(x)
        conf1   = torch.softmax(logits1, dim=1).max(dim=1).values
        if (conf1 >= threshold).all():
            exit_idx = torch.zeros(B, dtype=torch.long, device=x.device)
            return logits1, exit_idx

        x = self.layer2(x)
        logits2 = self.branch2(x)
        conf2   = torch.softmax(logits2, dim=1).max(dim=1).values
        if (conf2 >= threshold).all():
            exit_idx = torch.ones(B, dtype=torch.long, device=x.device)
            return logits2, exit_idx

        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        logits3 = self.fc(x)

        exit_idx = torch.full((B,), 2, dtype=torch.long, device=x.device)
        exit_idx[conf2 >= threshold] = 1
        exit_idx[conf1 >= threshold] = 0

        final_logits = logits3.clone()
        mask2 = conf2 >= threshold
        final_logits[mask2] = logits2[mask2]
        mask1 = conf1 >= threshold
        final_logits[mask1] = logits1[mask1]

        return final_logits, exit_idx


def build_model(num_classes=NUM_CLASSES):
    return BranchyResNet50(num_classes=num_classes, pretrained=False)


# ── Load the saved BranchyNet weights ────────────────────────────
model = build_model(NUM_CLASSES)
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model = model.to(DEVICE).eval()
total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Weights loaded from {SAVE_PATH}")
print(f"Total parameters (BranchyNet): {total_params:,}")

# ── Test loader (same CIFAR-10 test split used at training time) ─
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
test_set    = torchvision.datasets.CIFAR10(root='../data', train=False,
                                            download=True, transform=transform_test)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=128,
                                           shuffle=False, num_workers=0)

# ── FULL EVALUATION (final exit only) ────────────────────────────
print("\n" + "="*60)
print("FULL EVALUATION — final exit (no early stopping)")
print("="*60)

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(DEVICE)
        _, _, logits3 = model(inputs)
        all_preds.extend(logits3.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc       = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
recall    = recall_score(all_labels, all_preds, average='macro', zero_division=0)
f1        = f1_score(all_labels, all_preds, average='macro', zero_division=0)

print(f"  Accuracy          : {acc:.4f}")
print(f"  Precision (macro) : {precision:.4f}")
print(f"  Recall    (macro) : {recall:.4f}")
print(f"  F1-score  (macro) : {f1:.4f}")

# ── ADAPTIVE COMPUTATION EVALUATION (threshold sweep) ────────────
print("\n" + "="*60)
print("ADAPTIVE COMPUTATION — Early-Exit Threshold Sweep")
print("="*60)
print(f"  Thresholds tested: {EXIT_THRESHOLDS}")

adaptive_results = []
for tau in EXIT_THRESHOLDS:
    preds_list, labels_list, exit_idx_list = [], [], []

    t_start = time.time()
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(DEVICE)
            logits, exit_idx = model.adaptive_forward(inputs, threshold=tau)
            preds_list.extend(logits.argmax(1).cpu().numpy())
            labels_list.extend(labels.numpy())
            exit_idx_list.extend(exit_idx.cpu().numpy())
    t_end = time.time()

    preds_arr    = np.array(preds_list)
    labels_arr   = np.array(labels_list)
    exit_idx_arr = np.array(exit_idx_list)
    n            = len(labels_arr)

    acc_t  = accuracy_score(labels_arr, preds_arr)
    prec_t = precision_score(labels_arr, preds_arr, average='macro', zero_division=0)
    rec_t  = recall_score(labels_arr, preds_arr, average='macro', zero_division=0)
    f1_t   = f1_score(labels_arr, preds_arr, average='macro', zero_division=0)

    frac_exit1  = (exit_idx_arr == 0).mean()
    frac_exit2  = (exit_idx_arr == 1).mean()
    frac_exit3  = (exit_idx_arr == 2).mean()
    avg_time_ms = (t_end - t_start) / n * 1000

    adaptive_results.append({
        "threshold"  : tau,
        "accuracy"   : round(acc_t,  6),
        "precision"  : round(prec_t, 6),
        "recall"     : round(rec_t,  6),
        "f1"         : round(f1_t,   6),
        "frac_exit1" : round(frac_exit1, 4),
        "frac_exit2" : round(frac_exit2, 4),
        "frac_exit3" : round(frac_exit3, 4),
        "avg_time_ms": round(avg_time_ms, 4),
    })

    print(f"  τ={tau:.2f} | Acc={acc_t:.4f} | F1={f1_t:.4f} | "
          f"Exit1={frac_exit1:.1%} Exit2={frac_exit2:.1%} Exit3={frac_exit3:.1%} | "
          f"Time={avg_time_ms:.4f}ms/sample")

best_adaptive = max(adaptive_results, key=lambda r: r["accuracy"])

# ── FLOPs ─────────────────────────────────────────────────────
dummy_flops = torch.randn(1, 3, 32, 32).to(DEVICE)
macs, _     = thop_profile(model, inputs=(dummy_flops,), verbose=False)
flops_G     = macs * 2 / 1e9

# ── Parameters & disk size ───────────────────────────────────────
size_mb = os.path.getsize(SAVE_PATH) / 1e6

# # ── Inference time — CPU (full model, all 3 exits computed) ─────
# print("\n⏱  Measuring CPU inference times ...")
# model_cpu = build_model(NUM_CLASSES)
# model_cpu.load_state_dict(torch.load(SAVE_PATH, map_location="cpu"))
# model_cpu = model_cpu.eval()

# dummy_single_cpu = torch.randn(1, 3, 32, 32)
# dummy_batch_cpu  = torch.randn(128, 3, 32, 32)

# with torch.no_grad():
#     for _ in range(10):
#         model_cpu(dummy_single_cpu)

# times_cpu_single = []
# with torch.no_grad():
#     for _ in range(100):
#         t0 = time.perf_counter()
#         model_cpu(dummy_single_cpu)
#         times_cpu_single.append(time.perf_counter() - t0)
# inf_ms_single_cpu = np.mean(times_cpu_single) * 1000

# with torch.no_grad():
#     for _ in range(5):
#         model_cpu(dummy_batch_cpu)

# times_cpu_batch = []
# with torch.no_grad():
#     for _ in range(20):
#         t0 = time.perf_counter()
#         model_cpu(dummy_batch_cpu)
#         times_cpu_batch.append(time.perf_counter() - t0)
# inf_ms_batch128_cpu     = np.mean(times_cpu_batch) * 1000
# inf_ms_per_img_cpu      = inf_ms_batch128_cpu / 128
# throughput_imgs_sec_cpu = 128 / (inf_ms_batch128_cpu / 1000)

# del model_cpu, dummy_single_cpu, dummy_batch_cpu
# print("✓ CPU timing done")

# ── Inference time — GPU (full model, all 3 exits computed) ──────
for i in range(1):
    use_cuda = DEVICE.type == "cuda"
    print("⏱  Measuring GPU inference times ..." if use_cuda
        else "⏱  No CUDA device found — measuring 'GPU' timings on CPU instead ...")

    dummy_single_gpu = torch.randn(1, 3, 32, 32, device=DEVICE)

    with torch.no_grad():
        for _ in range(50):
            model(dummy_single_gpu)
    if use_cuda:
        torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for _ in range(500):
            if use_cuda:
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(dummy_single_gpu)
            if use_cuda:
                torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
    inf_ms_single_gpu = np.mean(times) * 1000

    dummy_batch_gpu = torch.randn(128, 3, 32, 32, device=DEVICE)

    if use_cuda:
        start_ev = torch.cuda.Event(enable_timing=True)
        end_ev   = torch.cuda.Event(enable_timing=True)
        with torch.no_grad():
            for _ in range(10):
                model(dummy_batch_gpu)
        torch.cuda.synchronize()

        batch_times_gpu = []
        with torch.no_grad():
            for _ in range(100):
                start_ev.record()
                model(dummy_batch_gpu)
                end_ev.record()
                torch.cuda.synchronize()
                batch_times_gpu.append(start_ev.elapsed_time(end_ev))
        inf_ms_batch128_gpu = np.mean(batch_times_gpu)
        gpu_timing_method = "CUDA events + torch.cuda.synchronize()"
    else:
        with torch.no_grad():
            for _ in range(5):
                model(dummy_batch_gpu)
        batch_times_gpu = []
        with torch.no_grad():
            for _ in range(20):
                t0 = time.perf_counter()
                model(dummy_batch_gpu)
                batch_times_gpu.append((time.perf_counter() - t0) * 1000)
        inf_ms_batch128_gpu = np.mean(batch_times_gpu)
        gpu_timing_method = "time.perf_counter() (no CUDA available)"

    inf_ms_per_img_gpu      = inf_ms_batch128_gpu / 128
    throughput_imgs_sec_gpu = 128 / (inf_ms_batch128_gpu / 1000)
    print(f"✓ GPU timing done for {i}")
    print(f"  --- Inference (GPU) ---")
    print(f"  Single image GPU  : {inf_ms_single_gpu:.3f} ms")
    print(f"  Batch-128 GPU     : {inf_ms_batch128_gpu:.2f} ms")
    print(f"  Per-image GPU     : {inf_ms_per_img_gpu:.3f} ms")
    print(f"  Throughput GPU    : {throughput_imgs_sec_gpu:.1f} imgs/sec")
    print("\n")
print("✓ Loop for GPU timing is done")

# ── Adaptive inference timing (τ=0.8, single image) ──────────────
with torch.no_grad():
    for _ in range(3):
        model.adaptive_forward(dummy_single_gpu, threshold=0.8)
    if use_cuda:
        torch.cuda.synchronize()
        start_evt = torch.cuda.Event(enable_timing=True)
        end_evt   = torch.cuda.Event(enable_timing=True)
        start_evt.record()
        for _ in range(50):
            model.adaptive_forward(dummy_single_gpu, threshold=0.8)
        end_evt.record()
        torch.cuda.synchronize()
        inf_ms_adapt_tau08 = start_evt.elapsed_time(end_evt) / 50
    else:
        t0 = time.perf_counter()
        for _ in range(50):
            model.adaptive_forward(dummy_single_gpu, threshold=0.8)
        inf_ms_adapt_tau08 = (time.perf_counter() - t0) / 50 * 1000

# ── Print results ─────────────────────────────────────────────
print(f"\n{'='*60}")
print("BRANCHYNET METRICS (recomputed from saved weights, CIFAR-10)")
print(f"{'='*60}")
print(f"  Accuracy (final exit) : {acc:.4f}")
print(f"  Precision (macro)     : {precision:.4f}")
print(f"  Recall    (macro)     : {recall:.4f}")
print(f"  F1-score  (macro)     : {f1:.4f}")
print(f"  Parameters            : {total_params:,}")
print(f"  Model size            : {size_mb:.2f} MB")
print(f"  FLOPs (full, all exits): {flops_G:.3f} GFLOPs")
# print(f"  --- Inference (CPU, full model) ---")
# print(f"  Single image      : {inf_ms_single_cpu:.3f} ms")
# print(f"  Batch-128         : {inf_ms_batch128_cpu:.2f} ms")
# print(f"  Throughput        : {throughput_imgs_sec_cpu:.1f} imgs/sec")
print(f"  --- Inference (GPU, full model) ---")
print(f"  Single image      : {inf_ms_single_gpu:.3f} ms")
print(f"  Batch-128         : {inf_ms_batch128_gpu:.2f} ms")
print(f"  Throughput        : {throughput_imgs_sec_gpu:.1f} imgs/sec")
print(f"  --- Adaptive (τ=0.8, single image) ---")
print(f"  Latency           : {inf_ms_adapt_tau08:.3f} ms")
print(f"\n  Best adaptive result (τ={best_adaptive['threshold']}):")
print(f"    Accuracy   : {best_adaptive['accuracy']:.4f}")
print(f"    Exit1      : {best_adaptive['frac_exit1']:.1%}")
print(f"    Exit2      : {best_adaptive['frac_exit2']:.1%}")
print(f"    Exit3      : {best_adaptive['frac_exit3']:.1%}")

# ── Save metrics JSON (same top-level schema as the baseline eval) ──
branchynet_metrics = {
    "accuracy"  : acc,
    "precision" : precision,
    "recall"    : recall,
    "f1"        : f1,
    "params"    : total_params,
    "size_mb"   : size_mb,
    "flops_G"   : flops_G,
    "inference_ms": {
        # "cpu": {
        #     "single_img_ms"      : round(inf_ms_single_cpu, 4),
        #     "batch128_ms"        : round(inf_ms_batch128_cpu, 4),
        #     "per_img_ms"         : round(inf_ms_per_img_cpu, 4),
        #     "throughput_imgs_sec": round(throughput_imgs_sec_cpu, 1),
        #     "timing_method"      : "time.perf_counter()",
        # },
        "gpu": {
            "single_img_ms"      : round(inf_ms_single_gpu, 4),
            "batch128_ms"        : round(inf_ms_batch128_gpu, 4),
            "per_img_ms"         : round(inf_ms_per_img_gpu, 4),
            "throughput_imgs_sec": round(throughput_imgs_sec_gpu, 1),
            "timing_method"      : gpu_timing_method,
        },
    },
    "method"                : "early_exit",
    "variant"               : "BranchyNet_ResNet50",
    "dataset"                : "CIFAR-10",
    "num_exits"              : 3,
    "exit_positions"         : ["after layer1 (256ch)", "after layer2 (512ch)", "final fc (2048ch)"],
    "exit_thresholds_tested" : EXIT_THRESHOLDS,
    "inference_ms_adaptive_tau08_single": round(inf_ms_adapt_tau08, 4),
    "adaptive_threshold_results"        : adaptive_results,
    "best_adaptive_result"              : best_adaptive,
    "saved_model_path": os.path.abspath(SAVE_PATH),
}

with open("__3__branchynet_metrics_v2.json", "w") as f:
    json.dump(branchynet_metrics, f, indent=2)
print(f"\n✓ Metrics saved → __3__branchynet_metrics_v2.json")

Using device: cuda
✓ Weights loaded from branchynet_resnet50_cifar10_corrected.pth
Total parameters (BranchyNet): 23,623,518


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")



FULL EVALUATION — final exit (no early stopping)
  Accuracy          : 0.9555
  Precision (macro) : 0.9555
  Recall    (macro) : 0.9555
  F1-score  (macro) : 0.9555

ADAPTIVE COMPUTATION — Early-Exit Threshold Sweep
  Thresholds tested: [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
  τ=0.50 | Acc=0.9084 | F1=0.9081 | Exit1=81.6% Exit2=13.4% Exit3=5.0% | Time=0.5178ms/sample
  τ=0.60 | Acc=0.9295 | F1=0.9293 | Exit1=72.7% Exit2=17.6% Exit3=9.7% | Time=0.5116ms/sample
  τ=0.70 | Acc=0.9440 | F1=0.9439 | Exit1=63.6% Exit2=20.4% Exit3=16.1% | Time=0.5137ms/sample
  τ=0.80 | Acc=0.9504 | F1=0.9504 | Exit1=52.3% Exit2=23.0% Exit3=24.7% | Time=0.5125ms/sample
  τ=0.90 | Acc=0.9547 | F1=0.9547 | Exit1=35.8% Exit2=22.1% Exit3=42.1% | Time=0.5137ms/sample
  τ=0.95 | Acc=0.9554 | F1=0.9554 | Exit1=21.4% Exit2=16.0% Exit3=62.6% | Time=0.5122ms/sample
⏱  Measuring GPU inference times ...
✓ GPU timing done for 0
  --- Inference (GPU) ---
  Single image GPU  : 4.945 ms
  Batch-128 GPU     : 52.03 ms
  Per-image G